# LTX-2.5 Alpha Gen — 영상에서 알파 매트 뽑기 (Kaggle T4 x2)

일반 영상을 넣으면 **알파 매트**(흰색 = 남길 부분, 검은색 = 지울 부분)를 생성해요.
그린스크린이나 마스크, 프롬프트 없이 머리카락, 연기, 유리 같은 반투명한 가장자리까지 잡는 게 강점이에요.
원본 워크플로우 `LTX_2.5_Alpha_Gen_Beta`를 T4에 맞게 옮긴 버전이에요.

**결과물:** 알파 매트 영상, **투명 배경 ProRes 4444 (.mov, AE·프리미어에 바로 사용)**, 그린 배경 합성, 프리멀티플라이, 비교 영상

**원본과 다른 점 (T4라서):** 22B 모델과 텍스트 인코더를 GGUF(4비트 압축)로 써요. 기본 크기는 짧은 변 512px, 97프레임(약 4초)이에요.
원본은 짧은 변 1088px, 최대 241프레임이에요.

**준비 (Kaggle 노트북 오른쪽 패널)**
1. Settings → Accelerator: **GPU T4 x2**, Internet: **On**
2. Add Input → 영상을 올린 **Dataset**을 추가하고, 설정 셀의 `INPUT_VIDEO` 경로를 맞춰요.
3. **꼭 필요:** LTX-2.5 VAE가 있는 [Lightricks/LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) 저장소는 승인제예요.
   Hugging Face에 로그인해서 저장소 페이지에서 사용 동의를 하고, Kaggle Add-ons → Secrets에 `HF_TOKEN`을 추가하세요.

**실행:** 위에서부터 순서대로 실행해요. 결과는 `/kaggle/working/alpha`에 저장돼요.

In [ ]:
# ============ 설정 (여기만 고치면 돼요) ============
INPUT_VIDEO = "/kaggle/input/my-clips/shot.mp4"   # 직접 찍었거나 사용 권한이 있는 영상

START_SEC = 0.0
MAX_FRAMES = 97          # 8n+1로 맞춰요. 97 ≈ 4초, 121 ≈ 5초, 145 ≈ 6초 (LoRA 공식 한계). T4에서는 97 이하 권장
SHORT_SIDE = 512         # 짧은 변 크기. 원본은 1088. T4에서는 512~640 권장 (32의 배수로 맞춰요)
FPS = None               # None: 원본 fps 유지. 숫자를 넣으면 그 fps로 바꿔요 (24, 25, 30 등)

# 모델 (GGUF는 커뮤니티 압축판이에요. 파일 목록을 보고 앞에 있는 것부터 골라요)
LTX_GGUF_REPO = "Abiray/LTX-2.5-Distilled-GGUF"
LTX_GGUF_PREFERENCE = ["Q4_K_M", "Q4_K_S", "Q3_K_M", "Q3_K_S"]
TE_GGUF_REPO = "elix3r/gemma4-12b-with-proj-ltx-2.5-GGUF"
TE_GGUF_PREFERENCE = ["Q4_K_M", "Q4_K_S", "Q5_K_M"]
TE_DEVICE = "cuda:1"     # 텍스트 인코더를 두 번째 T4에 올려요. 문제가 생기면 "default"
VAE_REPO = "Lightricks/LTX-2.5"                        # 승인제 (위 준비 3번)
LORA_REPO = "Lightricks/LTX-2.5-22b-IC-LoRA-Alpha-Gen" # 공식 Alpha Gen LoRA
LORA_STRENGTH = 1.0

# 샘플링 (원본 설정 그대로)
PROMPT = ""              # Alpha Gen은 프롬프트가 필요 없어요
NEGATIVE = ""
SEED = 42
CFG = 1.0
SAMPLER = "euler_ancestral"
SIGMAS = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
GUIDE_STRENGTH = 1.0
TILED_ENCODE = True      # 가이드 영상을 타일로 인코딩해서 VRAM을 아껴요

# 결과물
GREEN = (0, 177, 64)     # 그린 배경 색 (원본과 같은 #00B140)
SAVE_PRORES = True       # 투명 배경 ProRes 4444 .mov
SAVE_PNG_ZIP = False     # 투명 배경 PNG 시퀀스 (zip)

EXTRA_NODE_PACKS = {"ComfyUI-LTXVideo": "https://github.com/Lightricks/ComfyUI-LTXVideo"}
INPUT_FILES = [INPUT_VIDEO]

# T4는 bf16을 지원하지 않아서 fp16으로 강제해요. 영상이 검게 나오면 "--force-fp32"로 바꿔 보세요(느려요).
DTYPE_FLAG = "--force-fp16"
COMFY_ARGS = ["--lowvram", "--disable-smart-memory", "--cache-none", "--reserve-vram", "1.0", DTYPE_FLAG]

COMFY = "/tmp/ComfyUI"
MODEL_DIR = "/tmp/models"
OUT_DIR = "/kaggle/working/alpha"

## 1. 환경 확인

In [ ]:
import os, shutil, subprocess

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])
total, used, free = shutil.disk_usage("/tmp")
print(f"/tmp 여유 공간: {free / 1e9:.0f} GB (모델에 약 30GB 필요)")
with open("/proc/meminfo") as f:
    mem = {l.split(":")[0]: int(l.split()[1]) for l in f}
print(f"RAM: {mem['MemTotal'] / 1e6:.0f} GB, 사용 가능 {mem['MemAvailable'] / 1e6:.0f} GB")
if free < 32e9:
    print("⚠ /tmp 공간이 부족할 수 있어요. 다운로드가 실패하면 MODEL_VARIANT를 더 작은 것으로 바꾸세요.")
missing = [p for p in INPUT_FILES if not os.path.isfile(p)]
if missing:
    print("⚠ 입력 이미지를 찾을 수 없어요:", missing)
    print("  /kaggle/input 안의 파일:")
    for root, _, files in os.walk("/kaggle/input"):
        for fn in files[:20]:
            print("   ", os.path.join(root, fn))

## 2. ComfyUI와 커스텀 노드 설치

In [ ]:
import subprocess, sys, os

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if not os.path.isdir(COMFY):
    sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}")
else:
    sh(f"git -C {COMFY} pull --ff-only")

# 검증된 노드 팩만 사용해요 (출처가 불분명한 노드 팩은 설치하지 않아요)
NODE_PACKS = {
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes",      # MiniMax 저VRAM 패치
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF",           # GGUF 모델 로더
    "ComfyUI-MultiGPU": "https://github.com/pollockjj/ComfyUI-MultiGPU", # 텍스트 인코더를 두 번째 GPU로
    **EXTRA_NODE_PACKS,
}
for name, url in NODE_PACKS.items():
    path = f"{COMFY}/custom_nodes/{name}"
    if not os.path.isdir(path):
        sh(f"git clone --depth 1 {url} {path}")

sh(f"{sys.executable} -m pip install -q -r {COMFY}/requirements.txt")
for name in NODE_PACKS:
    req = f"{COMFY}/custom_nodes/{name}/requirements.txt"
    if os.path.isfile(req):
        sh(f"{sys.executable} -m pip install -q -r {req}")
sh(f"{sys.executable} -m pip install -q -U huggingface_hub gguf")

import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.device_count())

## 3. 모델 다운로드 (`/tmp/models`, 세션이 끝나면 지워져요)

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.errors import GatedRepoError, RepositoryNotFoundError

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not token:
    print("⚠ HF_TOKEN이 없어요. LTX-2.5 VAE는 승인제 저장소라 다운로드가 실패할 수 있어요.")

api = HfApi(token=token)

def pick(repo, includes, prefer=(), ext=".safetensors"):
    try:
        files = [f for f in api.list_repo_files(repo) if f.endswith(ext)]
    except (GatedRepoError, RepositoryNotFoundError) as e:
        raise RuntimeError(
            f"{repo}에 접근할 수 없어요. huggingface.co/{repo} 에서 사용 동의를 하고, "
            "Kaggle Secrets에 그 계정의 HF_TOKEN을 넣었는지 확인하세요.") from e
    hits = [f for f in files if all(s.lower() in f.lower() for s in includes)]
    for p in prefer:
        for f in hits:
            if p.lower() in f.lower():
                return f
    if hits:
        return hits[0]
    raise FileNotFoundError(f"{repo}에서 {includes} 파일을 못 찾았어요. 파일 목록:\n  " + "\n  ".join(files))

plan = {
    "unet": (LTX_GGUF_REPO, pick(LTX_GGUF_REPO, ["distilled"], LTX_GGUF_PREFERENCE, ext=".gguf"), "unet"),
    "te": (TE_GGUF_REPO, pick(TE_GGUF_REPO, ["gemma"], TE_GGUF_PREFERENCE, ext=".gguf"), "text_encoders"),
    "vae": (VAE_REPO, pick(VAE_REPO, ["video", "vae"], ["bf16"]), "vae"),
    "avae": (VAE_REPO, pick(VAE_REPO, ["audio", "vae"], ["bf16"]), "vae"),
    "lora": (LORA_REPO, pick(LORA_REPO, ["alpha"]), "loras"),
}

FILES = {}
for key, (repo, fname, folder) in plan.items():
    print(f"⬇ {key}: {repo}/{fname}")
    local = hf_hub_download(repo, fname, local_dir=f"{MODEL_DIR}/{repo.replace('/', '__')}", token=token)
    os.makedirs(f"{COMFY}/models/{folder}", exist_ok=True)
    link = f"{COMFY}/models/{folder}/{os.path.basename(fname)}"
    if not os.path.exists(link):
        os.symlink(local, link)
    FILES[key] = os.path.basename(fname)
    print(f"   {os.path.getsize(local) / 1e9:.1f} GB → models/{folder}/{FILES[key]}")
print(FILES)

## 4. 영상 준비 (크기·길이 맞추기)

In [ ]:
import glob, os, shutil, subprocess
import cv2
import numpy as np

FFMPEG = shutil.which("ffmpeg")
WORK = "/tmp/alpha_work"
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(f"{WORK}/frames")

cap = cv2.VideoCapture(INPUT_VIDEO)
src_fps, sw, sh = cap.get(cv2.CAP_PROP_FPS), cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cap.release()
VIDEO_FPS = float(FPS or round(src_fps, 3) or 24)
scale = f"scale=-2:{SHORT_SIDE}" if sw >= sh else f"scale={SHORT_SIDE}:-2"
subprocess.run([FFMPEG, "-y", "-loglevel", "error", "-ss", str(START_SEC), "-i", INPUT_VIDEO,
                "-vf", f"fps={VIDEO_FPS},{scale}", "-frames:v", str(MAX_FRAMES), "-start_number", "0",
                f"{WORK}/frames/%05d.png"], check=True)
paths = sorted(glob.glob(f"{WORK}/frames/*.png"))
n = (len(paths) - 1) // 8 * 8 + 1            # 8n+1 프레임으로 맞춰요 (LTX 요구 사항)
if n < 9:
    raise RuntimeError(f"영상이 너무 짧아요 ({len(paths)}프레임). 9프레임 이상 필요해요.")
ORIG = np.stack([cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB) for p in paths[:n]])
h0, w0 = ORIG.shape[1:3]

# 32의 배수가 되도록 오른쪽/아래를 가장자리 픽셀로 채워요 (늘리지 않고 패딩, 모델 카드 권장)
H = -(-h0 // 32) * 32
W = -(-w0 // 32) * 32
padded = np.pad(ORIG, ((0, 0), (0, H - h0), (0, W - w0), (0, 0)), mode="edge")

def encode(frames, path, extra_in=(), extra_out=(), pix_in="rgb24"):
    cmd = [FFMPEG, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", pix_in,
           "-s", f"{frames.shape[2]}x{frames.shape[1]}", "-r", str(VIDEO_FPS), "-i", "-", *extra_in, *extra_out, path]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    p.stdin.write(np.ascontiguousarray(frames).tobytes())
    p.stdin.close()
    if p.wait() != 0:
        raise RuntimeError(f"인코딩 실패: {path}")

PREP = f"{WORK}/prepared.mp4"
encode(padded, PREP, extra_out=["-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "10"])

AUDIO = f"{WORK}/audio.m4a"
r = subprocess.run([FFMPEG, "-y", "-loglevel", "error", "-ss", str(START_SEC), "-i", INPUT_VIDEO,
                    "-t", str(n / VIDEO_FPS), "-vn", "-c:a", "aac", AUDIO])
if r.returncode != 0 or not os.path.exists(AUDIO) or os.path.getsize(AUDIO) == 0:
    AUDIO = None
print(f"{n}프레임 ({n / VIDEO_FPS:.1f}초, {VIDEO_FPS}fps), 원본 {w0}x{h0} → 처리 {W}x{H}, 소리 {'있음' if AUDIO else '없음'}")

## 5. ComfyUI 서버 시작

In [ ]:
import subprocess, sys, time, json, urllib.request

BASE = "http://127.0.0.1:8188"
LOG = "/tmp/comfyui.log"

def server_up():
    try:
        urllib.request.urlopen(BASE + "/system_stats", timeout=5)
        return True
    except Exception:
        return False

if not server_up():
    server = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", *COMFY_ARGS],
        cwd=COMFY, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(180):
        if server_up() or server.poll() is not None:
            break
        time.sleep(5)

def log_tail(n=40):
    with open(LOG, errors="replace") as f:
        print("".join(f.readlines()[-n:]))

if server_up():
    print("✅ ComfyUI 실행 중")
    log_tail(15)
else:
    print("❌ ComfyUI가 시작되지 않았어요. 로그:")
    log_tail(80)

## 6. 워크플로우 도구 준비

In [ ]:
import json, math, os, shutil, time, uuid, urllib.request, urllib.error

def api(path, data=None):
    body = json.dumps(data).encode() if data is not None else None
    req = urllib.request.Request(BASE + path, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())

INFO = api("/object_info")

def combo_options(spec):
    t = spec[0]
    if isinstance(t, list):
        return t
    if t == "COMBO" and len(spec) > 1 and isinstance(spec[1], dict):
        return spec[1].get("options", [])
    return None

class Graph:
    """ComfyUI API 형식 그래프. 서버의 노드 정의로 입력 이름과 값을 검사해요."""
    def __init__(self):
        self.nodes, self.n = {}, 0

    def add(self, cls, **inputs):
        if cls not in INFO:
            raise RuntimeError(f"서버에 '{cls}' 노드가 없어요. 설치 셀 로그를 확인하세요.")
        req = INFO[cls]["input"].get("required", {})
        specs = {**req, **INFO[cls]["input"].get("optional", {})}
        for name, spec in req.items():
            if name in inputs or any(k.startswith(name + ".") for k in inputs):
                continue
            cfg = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
            opts = combo_options(spec)
            if opts:
                inputs[name] = cfg.get("default", opts[0])
            elif "default" in cfg:
                inputs[name] = cfg["default"]
            else:
                raise RuntimeError(f"{cls}: 필수 입력 '{name}'이 비어 있어요. 정의: {spec}")
        for name, value in inputs.items():
            if name not in specs:
                print(f"  · {cls}: 정의에 없는 입력 '{name}' (동적 입력이면 정상)")
                continue
            opts = combo_options(specs[name])
            if opts and isinstance(value, str) and value not in opts:
                raise RuntimeError(f"{cls}.{name}='{value}'가 선택지에 없어요: {opts[:15]}")
            kind = specs[name][0]
            if kind == "INT" and isinstance(value, float):
                inputs[name] = int(round(value))
            elif kind == "FLOAT" and isinstance(value, int) and not isinstance(value, bool):
                inputs[name] = float(value)
        self.n += 1
        nid = str(self.n)
        self.nodes[nid] = {"class_type": cls, "inputs": inputs}
        return nid

def out(nid, i=0):
    return [nid, i]

def video_size(aspect, megapixels):
    """원본 워크플로우의 해상도 표와 같은 크기 (32의 배수)."""
    ar = {"16:9": 16 / 9, "9:16": 9 / 16, "1:1": 1.0}[aspect]
    w = math.ceil(math.sqrt(megapixels * 1e6 * ar) / 32) * 32
    h = math.ceil(math.sqrt(megapixels * 1e6 / ar) / 32) * 32
    return w, h

def frame_count(seconds):
    """원본 워크플로우의 프레임 수 계산식."""
    f = max(5, round(seconds * 24))
    return f + (5 - f % 17) % 17

def add_input_file(g, path, name, node="LoadImage", field="image"):
    os.makedirs(f"{COMFY}/input", exist_ok=True)
    fname = f"{name}{os.path.splitext(path)[1]}"
    shutil.copy(path, f"{COMFY}/input/{fname}")
    INFO[node] = api(f"/object_info/{node}")[node]  # 새 파일이 선택지에 보이도록 갱신
    return out(g.add(node, **{field: fname}))

def add_input_image(g, path, name):
    return add_input_file(g, path, name)

def load_models(g, shift=None):
    """영상 모델, 텍스트 인코더, VAE를 불러오고 T4용 패치와 LoRA를 적용해요."""
    if MODEL_VARIANT == "gguf":
        model = out(g.add("UnetLoaderGGUF", unet_name=FILES["unet"]))
    else:
        model = out(g.add("UNETLoader", unet_name=FILES["unet"], weight_dtype="default"))
    if TE_DEVICE != "default" and "CLIPLoaderMultiGPU" in INFO:
        clip = out(g.add("CLIPLoaderMultiGPU", clip_name=FILES["te"], type="minimax", device=TE_DEVICE))
    else:
        if TE_DEVICE != "default":
            print("⚠ CLIPLoaderMultiGPU가 없어서 텍스트 인코더를 기본 GPU에 올려요.")
        clip = out(g.add("CLIPLoader", clip_name=FILES["te"], type="minimax"))
    vae = out(g.add("VAELoader", vae_name=FILES["vae"]))
    audio_vae = out(g.add("VAELoader", vae_name=FILES["avae"]))
    # 원본의 SageAttention/SolAttention은 T4에서 지원되지 않아 KJNodes 저VRAM 패치로 대신해요.
    if "MiniMaxChunkFeedForward" in INFO:
        model = out(g.add("MiniMaxChunkFeedForward", model=model, chunks=4, seq_threshold=2048))
    if "MiniMaxLowVRAMAttention" in INFO:
        model = out(g.add("MiniMaxLowVRAMAttention", model=model, head_chunks=4))
    if shift:
        model = out(g.add("MiniMaxH3SigmaShift", model=model, shift_video=shift[0], shift_audio=shift[1]))
    if TURBO_LORA:
        lora = g.add("LoraLoader", model=model, clip=clip, lora_name=FILES["lora"],
                     strength_model=1.0, strength_clip=1.0)
        model, clip = out(lora, 0), out(lora, 1)
    return model, clip, vae, audio_vae

def save_video(g, images, audio, fps, prefix):
    video = out(g.add("CreateVideo", images=images, fps=float(fps), audio=audio))
    g.add("SaveVideo", video=video, filename_prefix=prefix)

def run(g):
    with open("/kaggle/working/last_workflow_api.json", "w") as fp:
        json.dump(g.nodes, fp, indent=1, ensure_ascii=False)
    try:
        prompt_id = api("/prompt", {"prompt": g.nodes, "client_id": uuid.uuid4().hex})["prompt_id"]
    except urllib.error.HTTPError as e:
        print("❌ 워크플로우 오류:")
        print(json.dumps(json.loads(e.read()), indent=1, ensure_ascii=False)[:4000])
        raise
    start = time.time()
    while True:
        hist = api(f"/history/{prompt_id}")
        if prompt_id in hist:
            status = hist[prompt_id].get("status", {})
            if status.get("status_str") == "error":
                print("\n❌ 실행 중 오류:")
                for m in status.get("messages", []):
                    if m[0] == "execution_error":
                        print(m[1].get("node_type"), "-", m[1].get("exception_message"))
                log_tail(40)
                raise RuntimeError("generation failed")
            break
        time.sleep(15)
        lines = open(LOG, errors="replace").read().replace("\r", "\n").strip().splitlines()
        print(f"\r{(time.time() - start) / 60:.1f}분 경과 | {lines[-1][:110] if lines else ''}", end="", flush=True)
    print(f"\n✅ 완료: {(time.time() - start) / 60:.1f}분")

print(f"노드 정의 {len(INFO)}개 불러옴")

## 7. 알파 매트 생성

In [ ]:
import glob, time

g = Graph()
model = out(g.add("UnetLoaderGGUF", unet_name=FILES["unet"]))
lora = g.add("LTXICLoRALoaderModelOnly", model=model, lora_name=FILES["lora"], strength_model=LORA_STRENGTH)
model = out(lora, 0)
# IC-LoRA 로더가 알려 주는 잠재 축소 비율 (원본 워크플로우와 같은 연결)
downscale = out(lora, 1) if len(INFO["LTXICLoRALoaderModelOnly"].get("output", [])) > 1 else 1.0

if TE_DEVICE != "default" and "CLIPLoaderGGUFMultiGPU" in INFO:
    clip = out(g.add("CLIPLoaderGGUFMultiGPU", clip_name=FILES["te"], type="ltxv", device=TE_DEVICE))
else:
    clip = out(g.add("CLIPLoaderGGUF", clip_name=FILES["te"], type="ltxv"))
vae = out(g.add("VAELoader", vae_name=FILES["vae"]))
audio_vae = out(g.add("VAELoader", vae_name=FILES["avae"]))

pos = out(g.add("CLIPTextEncode", clip=clip, text=PROMPT))
neg = out(g.add("CLIPTextEncode", clip=clip, text=NEGATIVE))
cond = g.add("LTXVConditioning", positive=pos, negative=neg, frame_rate=VIDEO_FPS)

source = add_input_file(g, PREP, "alpha_src", node="LoadVideo", field="file")
frames_in = out(g.add("GetVideoComponents", video=source), 0)
empty = out(g.add("EmptyLTXVLatentVideo", width=W, height=H, length=n, batch_size=1))
guide = g.add("LTXAddVideoICLoRAGuide", positive=out(cond, 0), negative=out(cond, 1), vae=vae, latent=empty,
              image=frames_in, frame_idx=0, strength=GUIDE_STRENGTH, latent_downscale_factor=downscale,
              crop="disabled", use_tiled_encode=TILED_ENCODE)
audio_latent = out(g.add("LTXVEmptyLatentAudio", audio_vae=audio_vae, frames_number=n,
                         frame_rate=int(round(VIDEO_FPS)), batch_size=1))
av = out(g.add("LTXVConcatAVLatent", video_latent=out(guide, 2), audio_latent=audio_latent))

noise = out(g.add("RandomNoise", noise_seed=SEED))
guider = out(g.add("CFGGuider", model=model, positive=out(guide, 0), negative=out(guide, 1), cfg=CFG))
sampler = out(g.add("KSamplerSelect", sampler_name=SAMPLER))
sigmas = out(g.add("ManualSigmas", sigmas=SIGMAS))
sampled = g.add("SamplerCustomAdvanced", noise=noise, guider=guider, sampler=sampler, sigmas=sigmas, latent_image=av)
video_latent = out(g.add("LTXVSeparateAVLatent", av_latent=out(sampled, 0)), 0)
cropped = g.add("LTXVCropGuides", positive=out(guide, 0), negative=out(guide, 1), latent=video_latent)
matte = out(g.add("VAEDecodeTiled", samples=out(cropped, 2), vae=vae, tile_size=512, overlap=64,
                  temporal_size=64, temporal_overlap=8))
PREFIX = f"alpha/matte_{int(time.time())}"
g.add("SaveImage", images=matte, filename_prefix=PREFIX)
run(g)

## 8. 결과 만들기 (투명 배경, 그린 합성, 비교 영상)

In [ ]:
import glob, os, shutil, zipfile
import cv2
import numpy as np
from IPython.display import Video, display

files = sorted(glob.glob(f"{COMFY}/output/{PREFIX}_*.png"))
if not files:
    log_tail(40)
    raise RuntimeError("알파 매트 프레임이 저장되지 않았어요.")
m = np.stack([cv2.imread(p, cv2.IMREAD_GRAYSCALE) for p in files])[:, :h0, :w0]   # 패딩 잘라내기
count = min(len(m), len(ORIG))
m, rgb = m[:count], ORIG[:count]
a = (m.astype(np.float32) / 255.0)[..., None]
print(f"알파 매트 {count}프레임, 평균 불투명도 {a.mean() * 100:.1f}%")

os.makedirs(OUT_DIR, exist_ok=True)
name = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
stamp = time.strftime("%m%d_%H%M")
base = f"{OUT_DIR}/{name}_{stamp}"
h264 = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "16"]
with_audio = (["-i", AUDIO], ["-c:a", "copy", "-shortest"]) if AUDIO else ((), ())

matte_rgb = np.repeat(m[..., None], 3, axis=-1)
encode(matte_rgb, f"{base}_matte.mp4", extra_out=h264)
green = (rgb * a + np.array(GREEN, np.float32) * (1 - a)).astype(np.uint8)
encode(green, f"{base}_over_green.mp4", with_audio[0], [*with_audio[1], *h264])
premult = (rgb * a).astype(np.uint8)
encode(premult, f"{base}_premult.mp4", extra_out=h264)
if SAVE_PRORES:
    rgba = np.concatenate([rgb, m[..., None]], axis=-1)
    encode(rgba, f"{base}_rgba_prores4444.mov", with_audio[0],
           [*with_audio[1], "-c:v", "prores_ks", "-profile:v", "4444", "-pix_fmt", "yuva444p10le"], pix_in="rgba")
if SAVE_PNG_ZIP:
    with zipfile.ZipFile(f"{base}_rgba_png.zip", "w", zipfile.ZIP_STORED) as z:
        for i in range(count):
            ok, buf = cv2.imencode(".png", cv2.cvtColor(np.dstack([rgb[i], m[i]]), cv2.COLOR_RGBA2BGRA))
            z.writestr(f"{name}_{i:05d}.png", buf.tobytes())

# 확인용 2x2 (원본 | 매트 / 프리멀티 | 그린), 절반 크기 (원본 워크플로우의 contact sheet와 같은 배치)
half = lambda v: np.stack([cv2.resize(f, (w0 // 4 * 2, h0 // 4 * 2), interpolation=cv2.INTER_AREA) for f in v])
sheet = np.concatenate([np.concatenate([half(rgb), half(matte_rgb)], axis=2),
                        np.concatenate([half(premult), half(green)], axis=2)], axis=1)
encode(sheet, f"{base}_contactsheet.mp4", extra_out=h264)

for f in sorted(glob.glob(f"{base}_*")):
    print("저장:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")
display(Video(f"{base}_contactsheet.mp4", embed=True, width=900))

## 문제 해결

오류가 나면 **오류 메시지와 로그 마지막 부분**을 그대로 복사해서 Claude에게 붙여 주세요.
실행한 워크플로우는 `/kaggle/working/last_workflow_api.json`에 저장돼요.

| 증상 | 시도해 볼 것 |
|---|---|
| `Lightricks/LTX-2.5에 접근할 수 없어요` | 저장소 페이지에서 사용 동의 + Kaggle Secrets의 `HF_TOKEN` 확인 |
| `CUDA out of memory` | `SHORT_SIDE`를 448이나 384로, `MAX_FRAMES`를 65나 49로 |
| 커널이 죽음 (RAM 부족) | `LTX_GGUF_PREFERENCE`를 `Q3_K_S` 우선으로, `TE_DEVICE="cuda:1"` 유지 |
| 매트가 검정/노이즈 | `DTYPE_FLAG="--force-fp32"` (느려짐) |
| 가장자리가 흐릿함 | T4에서 해상도를 낮춘 영향이에요. 짧은 구간만 `SHORT_SIDE=640`으로 시도 |
| `서버에 '...' 노드가 없어요` | 2번 셀 로그에서 ComfyUI-LTXVideo 설치 오류 확인 |